# Параметрическое исследование динамики численности маргариток

**Цель:** Систематически исследовать влияние ключевых параметров модели (max_age и init_white)
на долгосрочную динамику популяций черных и белых маргариток. Для каждой комбинации
параметров запускается симуляция на 1000 шагов, собираются данные о численности
и строятся графики изменения популяций во времени.

## Инициализация проекта
Активируем окружение проекта DrWatson и загружаем необходимые библиотеки

In [1]:
using DrWatson
@quickactivate "project"
using Agents
using DataFrames
using CairoMakie

## Подключение модуля с моделью
В файле `src/daisyworld.jl` содержится определение самой модели:
- Типы агентов и их свойства (цвет, возраст, альбедо)
- Функции для инициализации мира, правила роста и размножения
- Параметры модели (солнечная постоянная, альбедо маргариток, диффузия тепла)

In [2]:
include(srcdir("daisyworld.jl"))

daisyworld (generic function with 1 method)

## Определение агрегирующих функций для сбора данных
Для отслеживания динамики популяции определим функции, которые подсчитывают
количество черных и белых маргариток в модели
- `black(a)`: возвращает `true`, если маргаритка черная
- `white(a)`: возвращает `true`, если маргаритка белая
- `adata`: список агрегаций, которые будут собираться на каждом шаге

In [3]:
black(a) = a.breed == :black
white(a) = a.breed == :white
adata = [(black, count), (white, count)]

2-element Vector{Tuple{Function, typeof(count)}}:
 (Main.var"##278".black, count)
 (Main.var"##278".white, count)

## Параметры эксперимента
Определяем словарь параметров. Векторные параметры (max_age, init_white) будут перебираться,
скалярные остаются фиксированными.
- max_age: максимальный возраст маргариток [25, 40]
- init_white: начальная доля белых маргариток [0.2, 0.8]
- init_black: начальная доля черных маргариток (фиксировано 0.2)
- albedo_white: 0.75 (белые отражают 75% тепла)
- albedo_black: 0.25 (черные отражают 25% тепла)
- scenario: :default — солнечная активность не меняется

In [4]:
param_dict = Dict(
    :griddims => (30, 30),
    :max_age => [25, 40],
    :init_white => [0.2, 0.8],
    :init_black => 0.2,
    :albedo_white => 0.75,
    :albedo_black => 0.25,
    :surface_albedo => 0.4,
    :solar_change => 0.005,
    :solar_luminosity => 1.0,
    :scenario => :default,
    :seed => 165,
)

Dict{Symbol, Any} with 11 entries:
  :solar_luminosity => 1.0
  :scenario         => :default
  :griddims         => (30, 30)
  :max_age          => [25, 40]
  :surface_albedo   => 0.4
  :init_white       => [0.2, 0.8]
  :albedo_white     => 0.75
  :init_black       => 0.2
  :solar_change     => 0.005
  :albedo_black     => 0.25
  :seed             => 165

## Генерация всех комбинаций параметров
Функция dict_list создает список всех возможных комбинаций параметров.
В данном случае: 2 значения max_age × 2 значения init_white = 4 комбинации.

In [5]:
params_list = dict_list(param_dict)

4-element Vector{Dict{Symbol, Any}}:
 Dict(:solar_luminosity => 1.0, :scenario => :default, :griddims => (30, 30), :max_age => 25, :surface_albedo => 0.4, :init_white => 0.2, :albedo_white => 0.75, :init_black => 0.2, :solar_change => 0.005, :albedo_black => 0.25…)
 Dict(:solar_luminosity => 1.0, :scenario => :default, :griddims => (30, 30), :max_age => 25, :surface_albedo => 0.4, :init_white => 0.8, :albedo_white => 0.75, :init_black => 0.2, :solar_change => 0.005, :albedo_black => 0.25…)
 Dict(:solar_luminosity => 1.0, :scenario => :default, :griddims => (30, 30), :max_age => 40, :surface_albedo => 0.4, :init_white => 0.2, :albedo_white => 0.75, :init_black => 0.2, :solar_change => 0.005, :albedo_black => 0.25…)
 Dict(:solar_luminosity => 1.0, :scenario => :default, :griddims => (30, 30), :max_age => 40, :surface_albedo => 0.4, :init_white => 0.8, :albedo_white => 0.75, :init_black => 0.2, :solar_change => 0.005, :albedo_black => 0.25…)

## Цикл по всем комбинациям параметров
Для каждой комбинации создаем модель, запускаем на 1000 шагов,
строим график динамики численности и сохраняем его.

In [6]:
for params in params_list
    model = daisyworld(;params...)

    agent_df, model_df = run!(model, 1000; adata)

    fig = Figure(size = (600, 400))
    ax = Axis(fig[1, 1], xlabel = "tick", ylabel = "daisy count")

    blackl = lines!(ax, agent_df[!, :time], agent_df[!, :count_black], color = :black)

    whitel = lines!(ax, agent_df[!, :time], agent_df[!, :count_white], color = :orange)


    Legend(fig[1, 2], [blackl, whitel], ["black", "white"], labelsize = 12)

    plt_name = savename("daisy-count", params) * ".png"

    save(plotsdir(plt_name), fig)
end

## Вывод
После выполнения цикла все графики сохранены в директории `plots/`.
Можно проанализировать, как изменение максимального возраста маргариток
и начальной доли белых особей влияет на устойчивость и динамику популяций.